# volcast on Google Colab — Chronos-2, zero-shot
#
This notebook runs the **real** volcast library with the **real** Chronos-2 model,
which your home machine cannot run (2-core Celeron, no GPU, no torch).
#
**What it does**
1. Clones `github.com/1Stellarmachinist/volcast`
2. Installs the `chronos` extra (pulls torch + chronos-forecasting)
3. Fetches real prices via yfinance
4. Runs `engine="chronos"` for a 5-day volatility forecast
5. Backtests **drift vs chronos** head to head on identical data and costs
#
**Runtime → Change runtime type → CPU is fine.** Chronos-2 is 120M params and
zero-shot, so inference is seconds, not training. A GPU is only worth selecting
if you want it faster.
#
**Big caveat, stated up front:** Colab CPU is not your machine. Speeds here do
not transfer back to Windows. And a better forecast does not automatically mean
a profitable strategy — the last cell checks that properly.

## 1. Get the code

In [ ]:
import os, subprocess, sys, pathlib

REPO = "https://github.com/1Stellarmachinist/volcast.git"
WORK = pathlib.Path("/content/volcast")

if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(WORK)], check=True)
else:
    subprocess.run(["git", "-C", str(WORK), "pull", "--ff-only"], check=False)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("cwd:", os.getcwd())
print("\n".join(sorted(p.name for p in WORK.iterdir() if not p.name.startswith("."))))

## 2. Install
`[chronos]` pulls torch + chronos-forecasting>=2.0. The `[api]`/`[dev]` extras are
not needed here. This is the step that never runs on Windows.

In [ ]:
!pip install -q -e ".[chronos]"
!pip install -q yfinance

import torch, chronos
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("chronos-forecasting:", getattr(chronos, "__version__", "unknown"))

## 3. Real price data
Uses the library's own cache/fallback logic. If Yahoo rate-limits the Colab IP,
`load_prices` silently falls back to synthetic data — so the cell **prints which
source was actually used** rather than letting you believe a fake result.

In [ ]:
from volcast.data import load_prices
from volcast.panel import build_panel
from volcast.config import Config

cfg = Config()
cfg.universe = ["SPY", "QQQ", "IWM", "AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA"]
cfg.horizon = 5
cfg.vol_window = 20

prices = load_prices(cfg.universe, period="2y", allow_synthetic=True,
                     cache_root=".volcast/cache")

# Detect whether we got real data: synthetic series start near exactly 100.0.
first = prices.iloc[0]
looks_synthetic = bool(((first > 95) & (first < 105)).all())
print(f"bars={len(prices)}  tickers={len(prices.columns)}  "
      f"{prices.index[0].date()} -> {prices.index[-1].date()}")
print("SOURCE:", "SYNTHETIC (Yahoo failed)" if looks_synthetic else "REAL Yahoo data")
if looks_synthetic:
    print("WARNING: results below are on synthetic data, not the market.")
prices.tail(3).round(2)

In [ ]:
panel = build_panel(prices, vol_window=cfg.vol_window)
print("panel:", panel.prices.shape, "returns:", panel.returns.shape)

## 4. Chronos-2 forecast
The engine feeds Chronos-2 **log-returns** and converts its q10/q90 spread into a
volatility estimate via `sigma = (q90 - q10) / (2 * 1.2816)`. Feeding returns
(not prices) means the spread is already in return units.
#
Timing is printed because this is the thing your CPU could not do.

In [ ]:
import time, pandas as pd
from volcast.forecast import forecast_panel

t0 = time.time()
fc_chronos = forecast_panel(
    panel.returns, horizon=cfg.horizon, window=cfg.vol_window,
    engine="chronos", prices=panel.prices, context_len=256,
)
elapsed = time.time() - t0
print(f"chronos forecast of {len(cfg.universe)} tickers took {elapsed:.1f}s")
fc_chronos.round(6)

In [ ]:
t0 = time.time()
fc_drift = forecast_panel(panel.returns, horizon=cfg.horizon, window=cfg.vol_window,
                          engine="drift")
drift_elapsed = time.time() - t0

merged = fc_chronos[["ticker", "sigma_h"]].rename(columns={"sigma_h": "chronos_sigma_h"}).merge(
    fc_drift[["ticker", "sigma_h"]].rename(columns={"sigma_h": "drift_sigma_h"}), on="ticker"
)
merged["ratio_chronos_over_drift"] = merged["chronos_sigma_h"] / merged["drift_sigma_h"]
print(f"drift took {drift_elapsed*1000:.1f}ms; chronos took {elapsed:.1f}s "
      f"({elapsed/max(drift_elapsed,1e-9):,.0f}x slower)")
merged.round(6)

### Which forecast is actually better?
Compute the **pinball loss** of each engine on the realised 5-day return over a
short rolling evaluation. Lower is better. This is the only honest way to say
whether Chronos-2 earns its cost.

In [ ]:
import numpy as np
from volcast.evaluate import mean_pinball

H = cfg.horizon
rets = panel.returns
eval_end = len(rets) - H
start = max(len(rets) - H - 20, 60)   # ~20 evaluations; keep it short for CPU

rows = []
for i in range(start, eval_end):
    hist = rets.iloc[:i]
    try:
        c = forecast_panel(hist, horizon=H, window=cfg.vol_window, engine="chronos",
                           prices=panel.prices.iloc[:i], context_len=256)
    except Exception as e:
        print("chronos failed:", e)
        break
    d = forecast_panel(hist, horizon=H, window=cfg.vol_window, engine="drift")

    realised = rets.iloc[i:i+H].sum()          # forward H-day log return
    for ticker in cfg.universe:
        try:
            rz = float(realised[ticker])
            cs = float(c.loc[c.ticker == ticker, "sigma_h"].iloc[0])
            ds = float(d.loc[d.ticker == ticker, "sigma_h"].iloc[0])
        except Exception:
            continue
        # sigma_h is a scale, not a quantile: score a 1-sigma band as the 0.8413
        # quantile on each side, which is what a volatility forecast implies.
        rows.append({
            "i": i, "ticker": ticker, "realised": rz,
            "chronos_lo": -cs, "chronos_hi": cs,
            "drift_lo": -ds, "drift_hi": ds,
        })

ev = pd.DataFrame(rows)
print(f"{len(ev)} forecast evaluations")
if len(ev):
    q = 0.8413
    c_loss = (mean_pinball(ev["realised"], ev["chronos_hi"], q) +
              mean_pinball(ev["realised"], ev["chronos_lo"], 1 - q)) / 2
    d_loss = (mean_pinball(ev["realised"], ev["drift_hi"], q) +
              mean_pinball(ev["realised"], ev["drift_lo"], 1 - q)) / 2
    print(f"mean pinball loss  chronos={c_loss:.6f}   drift={d_loss:.6f}")
    print("LOWER IS BETTER ->", "chronos wins" if c_loss < d_loss else "drift wins")
    gap = abs(c_loss - d_loss) / max(d_loss, 1e-12)
    print(f"relative gap: {gap:.1%}")
    print(f"n={len(ev)} evaluations. A gap under a few percent is unlikely to be real.")

## 5. Head-to-head backtest
Identical universe, costs and rebalance schedule; only the forecast engine
differs. Reported with standard errors, because a Sharpe without an SE is
decoration.

In [ ]:
from volcast.backtest import run_backtest
from volcast.evaluate import metrics

results = {}
for engine in ("drift", "chronos"):
    try:
        t0 = time.time()
        res = run_backtest(
            panel, target_vol=cfg.target_vol, max_weight=cfg.max_weight,
            cost_bps=cfg.cost_bps, horizon=cfg.horizon, rebalance=cfg.rebalance,
            engine=engine, test_days=126, vol_window=cfg.vol_window,
        )
        results[engine] = res
        m = metrics(res.returns)
        print(f"{engine:8s} took {time.time()-t0:6.1f}s  "
              f"Sharpe={m['sharpe']:.3f}  SE={m['sharpe_se']:.3f}  "
              f"ann_ret={m['ann_return']:.2%}  maxDD={m['max_drawdown']:.2%}  "
              f"costs={res.stats['total_cost']:.4%}")
    except Exception as e:
        print(f"{engine} FAILED: {type(e).__name__}: {e}")

### Verdict

In [ ]:
print("=" * 68)
for engine, res in results.items():
    m = metrics(res.returns)
    s, se = m["sharpe"], m["sharpe_se"]
    verdict = ("SIGNIFICANT" if abs(s) > 2 * se else "INDISTINGUISHABLE FROM ZERO")
    print(f"{engine:8s} Sharpe {s:6.3f} +/- {se:.3f}  -> {verdict}")
if len(results) == 2:
    a, b = metrics(results["drift"].returns)["sharpe"], metrics(results["chronos"].returns)["sharpe"]
    print(f"\nchronos minus drift Sharpe: {b - a:+.3f}")
    print("A difference much smaller than the standard errors above is noise.")
print("=" * 68)

## 6. If you want to keep going
- Longer horizon (`cfg.horizon = 20`) — Chronos-2 degrades gracefully; drift does not.
- Multivariate mode: `predict_df` accepts several target columns plus covariates,
  which is where Chronos-2 is strongest. volcast currently treats names
  independently; that is the most promising next change.
- Save the model to Drive so a restart does not re-download 500MB:
  `pipeline` is cached under `~/.cache/huggingface`.